# Grokking: Weight Averaging & Distillation Experiment

Reproduces the grokking experiment with:
- **Baseline**: single model trained on full data
- **4 Specialists**: each trained on 25% of the data
- **Weight Averaged**: average specialist weights, fine-tune on full data
- **Distilled**: knowledge distillation from specialists

In [1]:
!pip install -q pytorch_lightning mod sympy blobfile numpy pandas matplotlib

## Clone Repo

In [2]:
import os, sys

# Clone this repo - already has validation fix + split_n_ways + distillation metrics
REPO_DIR = '/kaggle/working/grok'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/yazankb/grok.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Repo at: {REPO_DIR}')

Cloning into '/kaggle/working/grok'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 133 (delta 71), reused 93 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 346.78 KiB | 4.39 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Repo at: /kaggle/working/grok


## Imports & GPU

In [3]:
import torch
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from grok.multi_training import train_multi_with_distillation, add_multi_args

GPU_ID = 0 if torch.cuda.is_available() else -1
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## Configuration


In [4]:
# Experiment config
FINAL_STEPS = 25000
N_MODELS = 4
SPECIALIST_STEPS = FINAL_STEPS // N_MODELS
RANDOM_SEED = 42

LOGDIR = '/kaggle/working/logs'
DATADIR = os.path.join(REPO_DIR, 'data')
EXPERIMENT_NAME = 'grokking_experiment'

print(f'Final steps: {FINAL_STEPS}')
print(f'Specialist steps: {SPECIALIST_STEPS} x {N_MODELS} = {SPECIALIST_STEPS * N_MODELS} total')

Final steps: 25000
Specialist steps: 6250 x 4 = 25000 total


## Build Hyperparameters

In [5]:
parser = add_multi_args()
hparams = parser.parse_args([])

hparams.logdir = LOGDIR
hparams.datadir = DATADIR
hparams.math_operator = "+"
hparams.train_data_pct = 50
hparams.n_models = N_MODELS
hparams.specialist_steps = SPECIALIST_STEPS
hparams.final_steps = FINAL_STEPS
hparams.distill_steps = FINAL_STEPS
hparams.distill_temperature = 1.0
hparams.distill_alpha = 0.5
hparams.experiment_name = EXPERIMENT_NAME
hparams.run_baseline = True
hparams.random_seed = RANDOM_SEED
hparams.gpu = GPU_ID
hparams.weight_decay = 0.1
hparams.max_lr = 5e-4
hparams.warmup_steps = 10
hparams.n_layers = 2
hparams.n_heads = 4
hparams.d_model = 128

print(f'Model: {hparams.n_layers}L-{hparams.n_heads}H-{hparams.d_model}D')
print(f'Operator: {hparams.math_operator} mod 97 | Train: {hparams.train_data_pct}%')

Model: 2L-4H-128D
Operator: + mod 97 | Train: 50%


## Run Experiment

This runs: 4 specialists -> weight average -> fine-tune -> distill -> baseline

In [ ]:
experiment_dir = train_multi_with_distillation(hparams)
print(f'Results: {experiment_dir}')


Multi-model grokking with DISTILLATION: 4 specialists
Operator: +  |  train_data_pct: 50%
Full training set: 4704 equations
Validation set: 4705 equations

Phase 1 — Specialist 1/4  (shard size: 1176)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


  Starting specialist 0 training for 6250 steps...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ transformer │ Transformer │  455 K │ train │     0 │
└───┴─────────────┴─────────────┴────────┴───────┴───────┘

...

[Step 22000] Training in progress...

[Step 23000] Training in progress...

[Step 24000] Training in progress...

`Trainer.fit` stopped: `max_steps=25000` reached.



Phase 4 — Knowledge Distillation

Distillation: T=1.0, alpha=0.5
  Step 500/25000: train_acc=0.00%, val_acc=0.91%, loss=3.2824
  [Distillation Step 1000/25000]
  Step 1000/25000: train_acc=7.81%, val_acc=0.06%, loss=3.0496
  Step 1500/25000: train_acc=17.19%, val_acc=0.28%, loss=2.9742
  [Distillation Step 2000/25000]
  Step 2000/25000: train_acc=43.75%, val_acc=0.94%, loss=2.7953
  Step 2500/25000: train_acc=32.81%, val_acc=1.11%, loss=2.8840
  [Distillation Step 3000/25000]
  Step 3000/25000: train_acc=28.12%, val_acc=1.91%, loss=2.9326
  Step 3500/25000: train_acc=34.38%, val_acc=5.87%, loss=2.7976
  [Distillation Step 4000/25000]
  Step 4000/25000: train_acc=50.00%, val_acc=15.88%, loss=2.7703
  Step 4500/25000: train_acc=79.69%, val_acc=51.35%, loss=2.6973
  [Distillation Step 5000/25000]
  Step 5000/25000: train_acc=82.81%, val_acc=67.74%, loss=2.6435
  Step 5500/25000: train_acc=89.06%, val_acc=69.48%, loss=2.6317
  [Distillation Step 6000/25000]
  Step 6000/25000: train_acc=95

## Load Metrics

In [ ]:
import os

def find_csv(base_dir, subdir):
    path = os.path.join(base_dir, subdir, 'lightning_logs', 'version_0', 'metrics.csv')
    return path if os.path.exists(path) else None

metrics = {}
for i in range(N_MODELS):
    p = find_csv(experiment_dir, f'specialist_{i}')
    if p: metrics[f'specialist_{i}'] = p

for name in ['merged_average', 'baseline']:
    p = find_csv(experiment_dir, name)
    if p: metrics[name] = p

# Distillation has custom CSV
dc = os.path.join(experiment_dir, 'distilled', 'distill_metrics.csv')
if os.path.exists(dc): metrics['distilled'] = dc

dvc = os.path.join(experiment_dir, 'distilled', 'distill_val_metrics.csv')
if os.path.exists(dvc): metrics['distilled_val'] = dvc

print('Found:', list(metrics.keys()))

# Load and inspect data
data = {}
for k, v in metrics.items():
    df = pd.read_csv(v)
    # Drop rows where step or target metric is NaN
    if 'val_accuracy' in df.columns:
        df = df.dropna(subset=['val_accuracy', 'step'])
    elif 'student_acc' in df.columns:
        df = df.dropna(subset=['student_acc', 'step'])
    elif 'full_train_acc' in df.columns:
        df = df.dropna(subset=['full_train_acc', 'step'])
    data[k] = df
    print(f'{k}: {len(df)} rows, cols: {list(df.columns)[:10]}')

## Plot: Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'baseline': '#2196F3', 'merged_average': '#4CAF50', 'distilled': '#9C27B0'}

# Validation Accuracy
ax = axes[0]
for i in range(N_MODELS):
    name = f'specialist_{i}'
    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:
        data[name].plot(x='step', y='val_accuracy', ax=ax, alpha=0.3, color='gray', legend=False)

for name, color in colors.items():
    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:
        data[name].plot(x='step', y='val_accuracy', ax=ax, label=name, color=color, legend=True)

if 'distilled_val' in data and len(data['distilled_val']) > 0:
    data['distilled_val'].plot(x='step', y='val_accuracy', ax=ax, label='distilled', color=colors['distilled'], legend=True)

ax.set_title('Validation Accuracy')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy %')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Training Accuracy
ax = axes[1]
for i in range(N_MODELS):
    name = f'specialist_{i}'
    if name in data and len(data[name]) > 0:
        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'
        if col in data[name].columns:
            data[name].plot(x='step', y=col, ax=ax, alpha=0.3, color='gray', legend=False)

for name, color in colors.items():
    if name in data and len(data[name]) > 0:
        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'
        if col in data[name].columns:
            data[name].plot(x='step', y=col, ax=ax, label=name, color=color, legend=True)

if 'distilled' in data and len(data['distilled']) > 0 and 'student_acc' in data['distilled'].columns:
    data['distilled'].plot(x='step', y='student_acc', ax=ax, label='distilled', color=colors['distilled'], legend=True)

ax.set_title('Training Accuracy')
ax.set_xlabel('Step')
ax.set_ylabel('Accuracy %')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/accuracy.png', dpi=150)
plt.show()

## Plot: Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Val Accuracy
ax = axes[0, 0]
for name, color in colors.items():
    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:
        data[name].plot(x='step', y='val_accuracy', ax=ax, label=name, color=color)
if 'distilled_val' in data and len(data['distilled_val']) > 0:
    data['distilled_val'].plot(x='step', y='val_accuracy', ax=ax, label='distilled', color=colors['distilled'])
ax.set_title('Validation Accuracy')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Val Loss (log)
ax = axes[0, 1]
for name, color in colors.items():
    if name in data and len(data[name]) > 0 and 'val_loss' in data[name].columns:
        data[name].plot(x='step', y='val_loss', ax=ax, label=name, color=color)
ax.set_title('Validation Loss')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Train Accuracy
ax = axes[1, 0]
for name, color in colors.items():
    if name in data and len(data[name]) > 0:
        col = 'full_train_acc' if 'full_train_acc' in data[name].columns else 'train_accuracy'
        if col in data[name].columns:
            data[name].plot(x='step', y=col, ax=ax, label=name, color=color)
if 'distilled' in data and len(data['distilled']) > 0 and 'student_acc' in data['distilled'].columns:
    data['distilled'].plot(x='step', y='student_acc', ax=ax, label='distilled', color=colors['distilled'])
ax.set_title('Training Accuracy')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)

# Final bar chart
ax = axes[1, 1]
final_acc = []
labels = []
bar_colors = []
for name, label, bar_color in [('baseline', 'Baseline', colors['baseline']), ('merged_average', 'Weight Avg', colors['merged_average']), ('distilled_val', 'Distilled', colors['distilled'])]:
    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:
        acc = float(data[name]['val_accuracy'].iloc[-1])
        final_acc.append(acc)
        labels.append(label)
        bar_colors.append(bar_color)
if final_acc:
    bars = ax.bar(labels, final_acc, color=bar_colors)
    ax.set_ylabel('Accuracy %')
    ax.set_title('Final Val Accuracy')
    for bar, acc in zip(bars, final_acc):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{acc:.1f}%', ha='center')
    ax.set_ylim(0, max(final_acc) * 1.2 if final_acc else 100)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Grokking: {hparams.math_operator} mod 97 | {N_MODELS} specialists | {FINAL_STEPS:,} steps')
plt.tight_layout()
plt.savefig('/kaggle/working/dashboard.png', dpi=150)
plt.show()

print('Plots saved to /kaggle/working/')

## Results

In [ ]:
print('='*50)
print('FINAL RESULTS')
print('='*50)
for name, label in [('baseline', 'Baseline'), ('merged_average', 'Weight Avg'), ('distilled_val', 'Distilled')]:
    if name in data and len(data[name]) > 0 and 'val_accuracy' in data[name].columns:
        acc = float(data[name]['val_accuracy'].iloc[-1])
        print(f'{label:15s}: {acc:.1f}%')

In [ ]:
import torch
print(torch.version.cuda, torch.cuda.get_device_name(0))
